# Phase 2 Architecture A v1

Shared-backbone, multi-head XGBoost baseline for the Phase 2 prompt profiling dataset.

This notebook is intentionally unexecuted. Upload it to Google Colab, upload the merged CSV, then run cells top to bottom.

## Architecture Overview

```text
Prompt
  -> MiniLM sentence embedding
  -> PCA embedding compression
  -> hand-crafted prompt features
  -> shared feature matrix
  -> separate XGBoost heads:
       d1-d5
       tier
       intent
       task_type
       reasoning_chain_detected
  -> rule-based research_signals
  -> confidence from model probabilities
```

This is the safest first Phase 2 baseline because it extends the successful Phase 1 v4 style without adding deep-learning fine-tuning.

In [1]:
# Colab setup
!pip -q install sentence-transformers xgboost

In [2]:
import ast
import json
import re
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

## Load Dataset

Upload `prompt_classifier_phase1_phase2_merged_cleaned.csv` to Colab, then set `DATA_PATH` below.

In [5]:
DATA_PATH = '/content/prompt_classifier_phase1_phase2_merged_cleaned.csv'

df = pd.read_csv(DATA_PATH)
print(df.shape)
df.head()

(2273, 24)


,id,prompt,intent,task_type,reasoning_chain_detected,d1,d2,d3,d4,d5,...,research_signals,confidence,low_confidence_flag,task_description,expected_answer,prompting_techniques,prompt_type,phrasing_style,domain,source
0,NaN,Imagine you are a science fiction author renow...,SYNTHETIC,generation,True,0.75,0.50,0.50,0.0,0.0,...,[],0.80,False,Develop a creative narrative about neural pros...,The ideal output would be a well-structured an...,"['ROLE_PROMPTING', 'TREE_OF_THOUGHTS']",CREATIVE_WRITING,NaN,NaN,phase2
1,NaN,You are an expert photography tutor. I want to...,ANALYTICAL,generation,True,0.50,0.75,0.50,0.0,0.0,...,[],0.80,False,Decode the logic behind this photography techn...,The ideal output would be a Python function th...,['CODE_PROMPTING'],CODE_EXPLANATION,NaN,NaN,phase2
2,NaN,You are a leading neuroscientist specializing ...,ANALYTICAL,reasoning,True,0.75,0.75,0.75,0.5,0.5,...,"[""scientific""]",0.90,False,Chat about recent developments in neural prost...,The ideal answer would begin with a brief defi...,"['ROLE_PROMPTING', 'CHAIN_OF_THOUGHT']",CONVERSATIONAL,NaN,NaN,phase2
3,NaN,I want to understand the basic human emotions....,FACTUAL,generation,False,0.00,0.50,0.50,0.0,0.0,...,[],0.95,False,Distinguish between various basic emotions tec...,"An ideal answer would first define emotion, th...","['CHAIN_OF_THOUGHT', 'CONTEXTUAL_PROMPTING']",COMPARISON,NaN,NaN,phase2
4,NaN,Here are examples of competitive exclusion. Ex...,ANALYTICAL,reasoning,True,0.75,0.50,0.50,0.0,0.5,...,[],0.80,False,Code a solution for theoretical ecology,The principle of competitive exclusion states ...,['ONE_SHOT_FEW_SHOT'],PROGRAMMING_CODE_GENERATION,NaN,NaN,phase2


In [6]:
VALID_SCORES = [0.0, 0.25, 0.5, 0.75, 1.0]
SCORE_COLS = ['d1', 'd2', 'd3', 'd4', 'd5']
VALID_INTENTS = ['FACTUAL', 'ANALYTICAL', 'SYNTHETIC', 'STRATEGIC']
VALID_TASK_TYPES = ['classification', 'generation', 'reasoning', 'coding', 'summarisation', 'formatting']

DIMENSION_LABELS = {
    'd1': 'Semantic Complexity',
    'd2': 'Domain Specificity',
    'd3': 'Output Formality',
    'd4': 'Research Dependency',
    'd5': 'Context Requirement',
}

def parse_research_signals(value):
    if pd.isna(value):
        return []
    if isinstance(value, list):
        return value
    try:
        parsed = json.loads(value)
    except Exception:
        try:
            parsed = ast.literal_eval(value)
        except Exception:
            return []
    return parsed if isinstance(parsed, list) else []

def complexity_score_from_dims(row):
    return (
        row['d1'] * 0.35 +
        row['d2'] * 0.20 +
        row['d3'] * 0.20 +
        row['d4'] * 0.15 +
        row['d5'] * 0.10
    )

def tier_from_score(score):
    if score < 0.40:
        return 'T1'
    if score < 0.70:
        return 'T2'
    return 'T3'

df['research_signals_parsed'] = df['research_signals'].apply(parse_research_signals)
df['computed_complexity_score'] = df.apply(complexity_score_from_dims, axis=1)
df['computed_tier'] = df['computed_complexity_score'].apply(tier_from_score)

print('Tier formula mismatches:', (df['computed_tier'] != df['tier']).sum())
print('Max score deviation:', np.abs(df['computed_complexity_score'] - df['complexity_score']).max())
print('\nTier counts:')
print(df['tier'].value_counts().sort_index())
print('\nIntent counts:')
print(df['intent'].value_counts())
print('\nTask type counts:')
print(df['task_type'].value_counts())

Tier formula mismatches: 0
Max score deviation: 1.1102230246251565e-16

Tier counts:
tier
T1     942
T2    1018
T3     313
Name: count, dtype: int64

Intent counts:
intent
ANALYTICAL    1306
FACTUAL        413
SYNTHETIC      380
STRATEGIC      174
Name: count, dtype: int64

Task type counts:
task_type
reasoning         1226
generation         766
summarisation      112
classification      90
coding              77
formatting           2
Name: count, dtype: int64


## Hand-Crafted Features

These are intentionally simple and inspectable. They complement the sentence embedding with explicit signals for length, structure, code, research, context, and output format.

In [7]:
RESEARCH_TERMS = [
    'latest', 'current', 'recent', 'market', 'pricing', 'vendor', 'regulation',
    'regulatory', 'compliance', 'benchmark', 'competitive', 'competitor', 'research'
]

CODE_TERMS = [
    'python', 'sql', 'json', 'yaml', 'api', 'function', 'script', 'debug',
    'stack trace', 'kubernetes', 'terraform', 'code'
]

FORMAL_OUTPUT_TERMS = [
    'table', 'json', 'schema', 'report', 'brief', 'plan', 'requirements',
    'document', 'matrix', 'checklist', 'step-by-step', 'structured'
]

REASONING_TERMS = [
    'analyze', 'compare', 'evaluate', 'assess', 'design', 'recommend', 'why',
    'how', 'tradeoff', 'risk', 'strategy', 'architecture'
]

CONTEXT_TERMS = [
    'attached', 'below', 'provided', 'given', 'context', 'transcript', 'logs',
    'document', 'file', 'dataset', 'csv', 'policy'
]

def count_terms(text, terms):
    text = text.lower()
    return sum(text.count(term) for term in terms)

def handcrafted_features(prompts):
    rows = []
    for prompt in prompts:
        text = str(prompt)
        lower = text.lower()
        words = re.findall(r'\b\w+\b', lower)
        sentences = re.split(r'[.!?]+', text)
        lines = [line for line in text.splitlines() if line.strip()]

        row = {
            'char_len': len(text),
            'word_count': len(words),
            'sentence_count': max(1, len([s for s in sentences if s.strip()])),
            'line_count': len(lines),
            'question_count': text.count('?'),
            'comma_count': text.count(','),
            'colon_count': text.count(':'),
            'semicolon_count': text.count(';'),
            'bullet_signal': int(bool(re.search(r'(^|\n)\s*[-*0-9]+[.)-]', text))),
            'has_role_prompt': int(bool(re.search(r'\byou are\b|\bact as\b', lower))),
            'has_step_request': int(bool(re.search(r'\bstep[- ]by[- ]step\b|\bfirst\b|\bsecond\b|\bfinally\b', lower))),
            'has_code_block': int('```' in text),
            'research_term_count': count_terms(lower, RESEARCH_TERMS),
            'code_term_count': count_terms(lower, CODE_TERMS),
            'formal_output_term_count': count_terms(lower, FORMAL_OUTPUT_TERMS),
            'reasoning_term_count': count_terms(lower, REASONING_TERMS),
            'context_term_count': count_terms(lower, CONTEXT_TERMS),
            'asks_for_comparison': int(any(term in lower for term in ['compare', 'versus', ' vs ', 'difference between'])),
            'asks_for_generation': int(any(term in lower for term in ['create', 'draft', 'write', 'generate', 'compose', 'build'])),
            'asks_for_summary': int(any(term in lower for term in ['summarize', 'summary', 'condense', 'tl;dr'])),
            'external_freshness_signal': int(any(term in lower for term in ['latest', 'current', 'recent', 'today', 'now'])),
        }
        rows.append(row)
    return pd.DataFrame(rows).fillna(0)

hand_df = handcrafted_features(df['prompt'])
hand_df.head()

,char_len,word_count,sentence_count,line_count,question_count,comma_count,colon_count,semicolon_count,bullet_signal,has_role_prompt,...,has_code_block,research_term_count,code_term_count,formal_output_term_count,reasoning_term_count,context_term_count,asks_for_comparison,asks_for_generation,asks_for_summary,external_freshness_signal
0,1058,158,11,4,0,12,0,0,0,1,...,0,0,1,0,1,0,0,0,0,1
1,531,82,9,1,0,5,0,0,0,1,...,0,0,5,0,2,0,0,0,0,0
2,525,70,5,1,0,5,0,0,0,1,...,0,1,0,0,0,0,0,0,0,1
3,237,38,4,1,0,3,0,0,0,0,...,0,0,0,1,1,0,0,0,0,0
4,713,107,12,1,0,8,2,0,0,0,...,0,2,0,2,1,0,0,0,0,1


## Shared Feature Matrix

MiniLM embeddings are reduced with PCA, then concatenated with hand-crafted features and scaled.

In [8]:
embedding_model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

prompts = df['prompt'].astype(str).tolist()
embeddings = embedding_model.encode(
    prompts,
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True,
)

pca = PCA(n_components=35, random_state=RANDOM_STATE)
emb_pca = pca.fit_transform(embeddings)

X_raw = np.hstack([emb_pca, hand_df.values])
scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

print('Embedding shape:', embeddings.shape)
print('PCA shape:', emb_pca.shape)
print('Hand feature shape:', hand_df.shape)
print('Final feature shape:', X.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/36 [00:00<?, ?it/s]

Embedding shape: (2273, 384)
PCA shape: (2273, 35)
Hand feature shape: (2273, 21)
Final feature shape: (2273, 56)


## Label Encoding

In [9]:
score_to_class = {score: idx for idx, score in enumerate(VALID_SCORES)}
class_to_score = {idx: score for score, idx in score_to_class.items()}

label_encoders = {}
targets = {}

for col in SCORE_COLS:
    targets[col] = df[col].map(score_to_class).astype(int).values

for col in ['tier', 'intent', 'task_type']:
    le = LabelEncoder()
    targets[col] = le.fit_transform(df[col].astype(str))
    label_encoders[col] = le
    print(col, list(le.classes_))

targets['reasoning_chain_detected'] = df['reasoning_chain_detected'].astype(bool).astype(int).values

tier ['T1', 'T2', 'T3']
intent ['ANALYTICAL', 'FACTUAL', 'STRATEGIC', 'SYNTHETIC']
task_type ['classification', 'coding', 'formatting', 'generation', 'reasoning', 'summarisation']


## Train/Validation Split

Use tier stratification because routing tier is the most important headline metric.

In [10]:
train_idx, val_idx = train_test_split(
    np.arange(len(df)),
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=df['tier'],
)

X_train, X_val = X[train_idx], X[val_idx]

print('Train size:', len(train_idx))
print('Val size:', len(val_idx))
print('\nValidation tier counts:')
print(df.iloc[val_idx]['tier'].value_counts().sort_index())

Train size: 1818
Val size: 455

Validation tier counts:
tier
T1    188
T2    204
T3     63
Name: count, dtype: int64


## Train Multi-Head XGBoost Models

In [11]:
def make_xgb(num_classes):
    objective = 'binary:logistic' if num_classes == 2 else 'multi:softprob'
    params = dict(
        n_estimators=250,
        max_depth=3,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        objective=objective,
    )
    if num_classes > 2:
        params['num_class'] = num_classes
    return XGBClassifier(**params)

heads = {}
head_classes = {
    'd1': 5,
    'd2': 5,
    'd3': 5,
    'd4': 5,
    'd5': 5,
    'tier': len(label_encoders['tier'].classes_),
    'intent': len(label_encoders['intent'].classes_),
    'task_type': len(label_encoders['task_type'].classes_),
    'reasoning_chain_detected': 2,
}

for name, num_classes in head_classes.items():
    print(f'Training {name}...')
    model = make_xgb(num_classes)
    y_train = targets[name][train_idx]
    model.fit(X_train, y_train)
    heads[name] = model

print('Done.')

Training d1...
Training d2...
Training d3...
Training d4...
Training d5...
Training tier...
Training intent...
Training task_type...
Training reasoning_chain_detected...
Done.


## Evaluation

In [14]:
def evaluate_head(name):
    y_true = targets[name][val_idx]
    y_pred = heads[name].predict(X_val)
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro')
    print(f'\n=== {name} ===')
    print('Accuracy:', round(acc, 4))
    print('Macro F1:', round(macro_f1, 4))

    # Determine the unique classes present in the validation data
    present_classes = np.unique(np.concatenate([y_true, y_pred]))

    if name in label_encoders:
        all_labels = list(label_encoders[name].classes_)
        # Map present integer indices back to string labels
        labels = [all_labels[i] for i in present_classes]
    elif name in SCORE_COLS:
        all_labels = [str(v) for v in VALID_SCORES]
        labels = [all_labels[i] for i in present_classes]
    else:
        labels = [str(c) for c in present_classes]

    cm = confusion_matrix(y_true, y_pred, labels=present_classes)
    print(pd.DataFrame(cm, index=labels, columns=labels))

    # Pass labels explicitly to classification_report to ensure alignment
    print(classification_report(y_true, y_pred, labels=present_classes, target_names=labels))

for head_name in heads:
    evaluate_head(head_name)


=== d1 ===
Accuracy: 0.6857
Macro F1: 0.6695
      0.0  0.25  0.5  0.75  1.0
0.0    60     1   16     8    0
0.25    8     9    0     2    1
0.5    19     0   94    32    2
0.75    1     0   38   129    5
1.0     1     0    2     7   20
              precision    recall  f1-score   support

         0.0       0.67      0.71      0.69        85
        0.25       0.90      0.45      0.60        20
         0.5       0.63      0.64      0.63       147
        0.75       0.72      0.75      0.74       173
         1.0       0.71      0.67      0.69        30

    accuracy                           0.69       455
   macro avg       0.73      0.64      0.67       455
weighted avg       0.69      0.69      0.68       455


=== d2 ===
Accuracy: 0.7407
Macro F1: 0.5819
      0.0  0.25  0.5  0.75  1.0
0.0     7     2   15     0    0
0.25    1    11   18     3    0
0.5     3     4  244    13    3
0.75    0     0   39    60    2
1.0     0     0    4    11   15
              precision    recall  

In [15]:
# Formula-derived tier from predicted d1-d5, compared against direct tier head.
pred_dim_classes = {col: heads[col].predict(X_val) for col in SCORE_COLS}
pred_dims = pd.DataFrame({col: [class_to_score[int(v)] for v in pred_dim_classes[col]] for col in SCORE_COLS})
pred_dims['complexity_score'] = pred_dims.apply(complexity_score_from_dims, axis=1)
pred_dims['derived_tier'] = pred_dims['complexity_score'].apply(tier_from_score)

direct_tier_pred = label_encoders['tier'].inverse_transform(heads['tier'].predict(X_val))
true_tier = df.iloc[val_idx]['tier'].values

print('Direct tier accuracy:', accuracy_score(true_tier, direct_tier_pred))
print('Formula-derived tier accuracy:', accuracy_score(true_tier, pred_dims['derived_tier']))
print('\nDirect tier confusion:')
print(pd.DataFrame(confusion_matrix(true_tier, direct_tier_pred), index=['T1', 'T2', 'T3'], columns=['T1', 'T2', 'T3']))
print('\nFormula-derived tier confusion:')
print(pd.DataFrame(confusion_matrix(true_tier, pred_dims['derived_tier']), index=['T1', 'T2', 'T3'], columns=['T1', 'T2', 'T3']))

Direct tier accuracy: 0.7978021978021979
Formula-derived tier accuracy: 0.7736263736263737

Direct tier confusion:
     T1   T2  T3
T1  149   38   1
T2   31  168   5
T3    3   14  46

Formula-derived tier confusion:
     T1   T2  T3
T1  147   40   1
T2   35  166   3
T3    3   21  39


## Rule-Based Research Signals

Architecture A starts with rule-based research signal extraction. This can be replaced later with a multi-label classifier if needed.

In [16]:
RESEARCH_SIGNAL_KEYWORDS = {
    'market_research': ['market', 'industry', 'trend', 'tam', 'sam', 'som'],
    'competitive_analysis': ['competitor', 'competitive', 'benchmark', 'rival'],
    'regulatory_compliance': ['regulation', 'regulatory', 'compliance', 'gdpr', 'hipaa', 'sox', 'eu ai act'],
    'security': ['security', 'vulnerability', 'threat', 'risk', 'iam', 'zero trust'],
    'cloud_infrastructure': ['aws', 'azure', 'gcp', 'cloud', 'kubernetes', 'terraform'],
    'finops': ['finops', 'cost', 'spend', 'budget', 'showback', 'chargeback'],
    'devops': ['ci/cd', 'pipeline', 'deployment', 'sre', 'devops', 'observability'],
    'data_engineering': ['data pipeline', 'etl', 'warehouse', 'lakehouse', 'spark'],
    'ai_governance': ['ai governance', 'llm', 'model risk', 'genai', 'guardrail'],
    'system_integration': ['integration', 'api', 'webhook', 'middleware'],
    'supply_chain': ['supply chain', 'inventory', 'procurement', 'logistics'],
    'hr_tech': ['hr', 'employee', 'workforce', 'talent', 'recruiting'],
    'vendor_analysis': ['vendor', 'rfi', 'rfp', 'procurement'],
}

def extract_research_signals(prompt, d4_score):
    if d4_score <= 0:
        return []
    text = prompt.lower()
    signals = []
    for signal, keywords in RESEARCH_SIGNAL_KEYWORDS.items():
        if any(keyword in text for keyword in keywords):
            signals.append(signal)
    return signals if signals else ['external_research']

## Inference Function

In [17]:
def build_features_for_prompts(new_prompts):
    new_embeddings = embedding_model.encode(
        [str(p) for p in new_prompts],
        batch_size=64,
        show_progress_bar=False,
        normalize_embeddings=True,
    )
    new_emb_pca = pca.transform(new_embeddings)
    new_hand = handcrafted_features(new_prompts)
    new_raw = np.hstack([new_emb_pca, new_hand.values])
    return scaler.transform(new_raw)

def max_probability(model, X_one):
    proba = model.predict_proba(X_one)[0]
    return float(np.max(proba))

def predict_prompt(prompt):
    X_one = build_features_for_prompts([prompt])

    predicted_dims = {}
    confidences = []
    for col in SCORE_COLS:
        pred_class = int(heads[col].predict(X_one)[0])
        predicted_dims[col] = class_to_score[pred_class]
        confidences.append(max_probability(heads[col], X_one))

    score = (
        predicted_dims['d1'] * 0.35 +
        predicted_dims['d2'] * 0.20 +
        predicted_dims['d3'] * 0.20 +
        predicted_dims['d4'] * 0.15 +
        predicted_dims['d5'] * 0.10
    )

    direct_tier = label_encoders['tier'].inverse_transform(heads['tier'].predict(X_one))[0]
    intent = label_encoders['intent'].inverse_transform(heads['intent'].predict(X_one))[0]
    task_type = label_encoders['task_type'].inverse_transform(heads['task_type'].predict(X_one))[0]
    reasoning_chain = bool(heads['reasoning_chain_detected'].predict(X_one)[0])

    for col in ['tier', 'intent', 'task_type', 'reasoning_chain_detected']:
        confidences.append(max_probability(heads[col], X_one))

    # Conservative confidence for multi-head output.
    confidence = float(np.min(confidences))

    result = {}
    for col in SCORE_COLS:
        result[col] = predicted_dims[col]
        result[f'{col}_label'] = DIMENSION_LABELS[col]

    result.update({
        'complexity_score': round(float(score), 4),
        'tier': direct_tier,
        'formula_tier': tier_from_score(score),
        'intent': intent,
        'task_type': task_type,
        'reasoning_chain_detected': reasoning_chain,
        'research_signals': extract_research_signals(prompt, predicted_dims['d4']),
        'confidence': round(confidence, 4),
    })
    return result

In [18]:
sample_prompt = 'Design a multi-cloud GenAI governance architecture for a Fortune 500 company, including compliance risks and vendor evaluation criteria.'
print(json.dumps(predict_prompt(sample_prompt), indent=2))

{
  "d1": 1.0,
  "d1_label": "Semantic Complexity",
  "d2": 0.75,
  "d2_label": "Domain Specificity",
  "d3": 1.0,
  "d3_label": "Output Formality",
  "d4": 0.75,
  "d4_label": "Research Dependency",
  "d5": 0.25,
  "d5_label": "Context Requirement",
  "complexity_score": 0.8375,
  "tier": "T3",
  "formula_tier": "T3",
  "intent": "STRATEGIC",
  "task_type": "reasoning",
  "reasoning_chain_detected": true,
  "research_signals": [
    "regulatory_compliance",
    "security",
    "cloud_infrastructure",
    "ai_governance",
    "vendor_analysis"
  ],
  "confidence": 0.4248
}


## Next Experiments

- Add class/sample weights if minority classes underperform.
- Compare direct `tier` head vs formula-derived tier from predicted D-scores.
- Replace rule-based `research_signals` with a multi-label classifier once signal quality is audited.
- Add samples for `sparql_generation` and `formatting` if those task types matter in production.